# 📍 Chapter 2 — Acquiring Urban Data from OSM
## Python for Urban Analysis — Ghana Edition

---

> **Professor's note:**
> *"Before you can analyse a city, you must ACQUIRE it. OSM is the largest free geographic database on Earth. Learning to query it efficiently is the single most transferable skill in urban data science."*

### Learning Objectives
1. Pull administrative boundaries at multiple levels
2. Extract Points of Interest (POI) — hospitals, schools, markets, etc.
3. Extract polygon features — parks, leisure, water bodies
4. Understand OSM tags and how to compose custom queries
5. Visualise POI density and spatial distribution

### Ghana Context
Greater Accra Region has **29 MMDAs** (Metropolitan, Municipal, and District Assemblies). Accra's urban core is well-mapped on OSM thanks to HOT mapathons and local mapping communities.

## 2.0 — Install & Configure

In [ ]:
!pip install -q osmnx geopandas contextily folium mapclassify

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import osmnx as ox
import geopandas as gpd
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import contextily as cx
import folium
from pathlib import Path

# ── Configuration ─────────────────────────────────────────────
STUDY_AREA       = 'Accra, Ghana'
CRS_WGS84        = 'EPSG:4326'
CRS_WEB_MERCATOR = 'EPSG:3857'
CRS_UTM_ACCRA    = 'EPSG:32630'
DARK_BG          = '#0e1117'
AMENITY_COLOR    = '#00e5ff'

DATA_DIR    = Path('data')
OUTPUT_DIR  = Path('outputs')
FIGURES_DIR = OUTPUT_DIR / 'figures'
for d in [DATA_DIR, OUTPUT_DIR, FIGURES_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# ── OSM tag dictionaries ─────────────────────────────────────
AMENITY_TAGS = {
    'amenity': [
        'hospital', 'clinic', 'pharmacy',
        'school', 'university', 'library',
        'marketplace', 'bank', 'atm',
        'police', 'fire_station',
        'place_of_worship', 'restaurant', 'fuel',
    ]
}

LEISURE_TAGS = {
    'leisure': ['park', 'playground', 'sports_centre', 'garden', 'stadium']
}

print(f'📍 Study area: {STUDY_AREA}')

## 2.1 — Administrative Boundary

We reload the study-area boundary. This is the same pattern from Chapter 1 — everything starts with the boundary.

In [ ]:
print('📍 Pulling study-area boundary ...')
boundary = ox.geocode_to_gdf(STUDY_AREA)
print(f'   {STUDY_AREA} → {boundary.geometry.iloc[0].geom_type}')
print(f'   Bounding box: {boundary.total_bounds}')

## 2.2 — Points of Interest (POIs)

We use `ox.features_from_place()` with **tag dictionaries** to query specific feature types. OSM uses a `key=value` system:

| Feature | OSM Tag | Accra Examples |
|---------|---------|----------------|
| Hospital | `amenity=hospital` | Korle Bu, 37 Military Hospital |
| School | `amenity=school` | Achimota School, Presec |
| Market | `amenity=marketplace` | Makola Market, Kaneshie |
| Pharmacy | `amenity=pharmacy` | — |
| Fuel station | `amenity=fuel` | Shell, Total stations |
| Church/Mosque | `amenity=place_of_worship` | — |

> **Note:** OSMnx returns **mixed geometry types** (points, polygons, multipolygons). We convert everything to point geometry (centroid) for consistent mapping.

In [ ]:
print('🏥 Pulling amenities ...')
amenities = ox.features_from_place(STUDY_AREA, tags=AMENITY_TAGS)

# Unify to point geometry (centroid) for mapping
amenities_pts = amenities.copy()
amenities_pts['geometry'] = amenities_pts.geometry.centroid
amenities_pts = amenities_pts[amenities_pts.geometry.within(
    boundary.geometry.iloc[0]
)]

print(f'   Total amenities in study area: {len(amenities_pts)}')

# Breakdown by type
if 'amenity' in amenities_pts.columns:
    print('\n   ── Amenity breakdown ──')
    counts = amenities_pts['amenity'].value_counts().head(15)
    for cat, n in counts.items():
        print(f'      {cat:25s}  {n:>5}')

In [ ]:
# Save amenities
amenities_path = DATA_DIR / 'amenities.gpkg'
amenities_pts.to_file(amenities_path, driver='GPKG')
print(f'💾 Amenities saved → {amenities_path}')

## 2.3 — Leisure & Green Spaces

Parks, playgrounds, sports centres, and gardens. These are critical for the livability index in Chapter 6.

In [ ]:
print('🌳 Pulling leisure/green spaces ...')
try:
    leisure = ox.features_from_place(STUDY_AREA, tags=LEISURE_TAGS)
    leisure_polys = leisure[leisure.geometry.geom_type.isin(
        ['Polygon', 'MultiPolygon']
    )]
    print(f'   Leisure polygons found: {len(leisure_polys)}')
    leisure_path = DATA_DIR / 'leisure.gpkg'
    leisure_polys.to_file(leisure_path, driver='GPKG')
    print(f'💾 Leisure saved → {leisure_path}')
except Exception as e:
    print(f'   ⚠ No leisure features found or error: {e}')
    leisure_polys = gpd.GeoDataFrame()

## 2.4 — Visualise: POI Map on Dark Basemap

Colour-coding by amenity category lets you see the **spatial distribution** of services across the city. Notice how hospitals cluster differently from schools or markets.

> **Design note:** We use a curated colour palette where warm colours = health, cool colours = education, yellow = commerce.

In [ ]:
# Colour palette for amenity categories
category_colors = {
    'hospital':         '#ff1744',
    'clinic':           '#ff5252',
    'pharmacy':         '#ff8a80',
    'school':           '#448aff',
    'university':       '#2962ff',
    'library':          '#82b1ff',
    'marketplace':      '#ffc400',
    'bank':             '#69f0ae',
    'atm':              '#b9f6ca',
    'police':           '#e040fb',
    'fire_station':     '#ea80fc',
    'place_of_worship': '#b388ff',
    'restaurant':       '#ff9100',
    'fuel':             '#ffab40',
}

fig, ax = plt.subplots(1, 1, figsize=(14, 14))
fig.patch.set_facecolor(DARK_BG)
ax.set_facecolor(DARK_BG)

# Boundary outline
boundary.to_crs(CRS_WEB_MERCATOR).plot(
    ax=ax, facecolor='none', edgecolor='white', linewidth=2
)

# Plot each amenity category
if 'amenity' in amenities_pts.columns:
    legend_handles = []
    for cat, color in category_colors.items():
        subset = amenities_pts[amenities_pts['amenity'] == cat]
        if len(subset) > 0:
            subset.to_crs(CRS_WEB_MERCATOR).plot(
                ax=ax, color=color, markersize=12, alpha=0.8, zorder=5
            )
            legend_handles.append(
                mpatches.Patch(color=color, label=f'{cat} ({len(subset)})')
            )

    ax.legend(
        handles=legend_handles, loc='lower left',
        fontsize=7, facecolor='#1a1a2e', edgecolor='white',
        labelcolor='white', framealpha=0.9, ncol=2
    )

cx.add_basemap(ax, source=cx.providers.CartoDB.DarkMatter)
ax.set_axis_off()
ax.set_title(
    f'Points of Interest — {STUDY_AREA}',
    fontsize=16, fontweight='bold', color='white', pad=20
)

fig.tight_layout()
fig.savefig(FIGURES_DIR / 'ch2_poi_map.png', dpi=150, bbox_inches='tight', facecolor=DARK_BG)
plt.show()

## 2.5 — Interactive POI Map (Folium)

The interactive map lets you click on individual POIs to see their names and categories.

In [ ]:
centroid = boundary.geometry.iloc[0].centroid
m = folium.Map(
    location=[centroid.y, centroid.x],
    zoom_start=13,
    tiles='CartoDB dark_matter',
)

# Add boundary
folium.GeoJson(
    boundary.to_json(),
    style_function=lambda x: {
        'fillColor': 'transparent', 'color': '#00e5ff', 'weight': 2
    },
).add_to(m)

# Add amenity markers (sample for browser performance)
sample = amenities_pts.head(500)
for _, row in sample.iterrows():
    cat = row.get('amenity', 'unknown')
    name = row.get('name', cat)
    color = 'red' if cat in ['hospital', 'clinic'] else \
            'blue' if cat in ['school', 'university'] else \
            'orange' if cat == 'marketplace' else 'gray'
    folium.CircleMarker(
        location=[row.geometry.y, row.geometry.x],
        radius=4, color=color, fill=True,
        popup=f'<b>{name}</b><br>{cat}',
        tooltip=str(name),
    ).add_to(m)

m.save(str(OUTPUT_DIR / 'ch2_poi_interactive.html'))
m  # Display in notebook

## ✅ Chapter 2 Complete

### What you acquired:
- **Administrative boundary** for Accra
- **Amenity POIs** — hospitals, schools, markets, banks, fuel stations, etc.
- **Leisure/green spaces** — parks, playgrounds, stadiums

### Key takeaway:
> OSM's tag system lets you query any type of urban feature. The `ox.features_from_place()` function + a tag dictionary is your universal data acquisition tool.

### Next → Chapter 3: Building Footprints and Morphology